# Day3_02B_Parallel_Multi_Agent_Execution

## OpenAI Agents SDK - Advanced Agent Pattern

**Duration:** 20 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  
**Theme:** Running multiple specialist Agents at the same time

---

## Learning Objectives

By the end of this notebook, participants will be able to:

- Understand why parallel execution matters in multi-agent systems.
- Explain the difference between sequential and parallel Agent execution.
- Run multiple specialist Agents using `asyncio.gather`.
- Build a DSU-style AI Faculty Assistant that prepares multiple outputs faster.
- Relate parallel execution to real-world enterprise AI workflows.

---

## Previous Notebook Recap

In the previous notebook, we learned two important collaboration patterns:

1. **Handoff**  
   Route the user to the right specialist.

2. **Agent-as-Tool**  
   Use multiple specialist Agents internally and combine their outputs.

Now we will improve the Agent-as-Tool pattern.

Instead of running Agents one after another, we will run them **in parallel**.


# 1. Real-World Story: Preparing an FDP Session

Imagine a faculty coordinator asks:

> "Prepare tomorrow's FDP session on Agentic AI."

Three people can work on this:

- One person prepares the concept explanation.
- One person prepares quiz questions.
- One person prepares a lab demo.

Should they work one after another?

Not necessary.

They can work at the same time.

That is parallel execution.

---

## Sequential Work

```text
Explainer finishes
   ↓
Quiz Creator starts
   ↓
Lab Demo Creator starts
   ↓
Final material is ready
```

This is slower.

---

## Parallel Work

```text
Explainer starts ─┐
Quiz Creator starts ─┼── Combined Teaching Pack
Lab Demo Creator starts ─┘
```

This is faster.


# 2. Why Parallel Execution Matters

In real Agentic AI systems, multiple Agents may perform independent tasks.

Examples:

- Research Agent summarizes papers.
- Data Agent checks datasets.
- Compliance Agent reviews policy.
- Presentation Agent creates slide outline.

If these tasks do not depend on each other, they can run at the same time.

This improves:

- Speed
- User experience
- System responsiveness
- Productivity

---

## Simple Rule

If Task B depends on Task A, run sequentially.

If Task A, B, and C are independent, run them in parallel.


# Pause & Reflect

Ask the class:

> If three faculty members are preparing different parts of the same FDP session, should they wait for each other?

No.

They can work independently and combine later.

That is exactly what we are going to implement.


# 3. Setup

We will use:

- `Agent`
- `Runner`
- Python `asyncio`

Important:

In VS Code/Jupyter notebooks, use:

```python
await Runner.run(...)
```

instead of:

```python
Runner.run_sync(...)
```


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
import asyncio
import time

from agents import Agent, Runner

# 4. Load Environment Variables

This uses the standard root `.env` loading pattern.


In [ ]:
current = Path.cwd().resolve()

for folder in [current] + list(current.parents):
    env = folder / ".env"
    if env.exists():
        load_dotenv(env)
        break

print("API Key Loaded:", os.getenv("OPENAI_API_KEY") is not None)

# 5. Create Specialist Agents

We will create three Agents for preparing teaching material:

1. **Concept Explainer Agent**
2. **Quiz Creator Agent**
3. **Lab Demo Agent**

Each Agent works independently.


In [ ]:
concept_agent = Agent(
    name="Concept Explainer Agent",
    instructions="""
    You are a concept explanation specialist for engineering faculty.
    Explain topics in simple, classroom-friendly language.
    Keep the explanation concise and beginner-friendly.
    """,
)

quiz_agent = Agent(
    name="Quiz Creator Agent",
    instructions="""
    You are a quiz creation specialist.
    Create short classroom quiz questions with answers.
    Keep the questions suitable for engineering faculty training.
    """,
)

lab_demo_agent = Agent(
    name="Lab Demo Agent",
    instructions="""
    You are a lab demonstration specialist.
    Suggest practical VS Code notebook demos.
    Keep demos simple, interactive, and suitable for minimal coding background.
    """,
)

# 6. Sequential Execution First

Before learning parallel execution, let us first run the Agents sequentially.

Sequential means:

1. First Agent completes.
2. Then second Agent starts.
3. Then third Agent starts.

This is easy to understand, but slower.


In [ ]:
async def prepare_session_sequential(topic: str):
    """
    Runs specialist Agents one after another.
    """

    start_time = time.time()

    concept = await Runner.run(
        concept_agent,
        f"Explain {topic} in 5 simple bullet points."
    )

    quiz = await Runner.run(
        quiz_agent,
        f"Create 3 MCQs with answers on {topic}."
    )

    demo = await Runner.run(
        lab_demo_agent,
        f"Suggest one short notebook demo for {topic}."
    )

    end_time = time.time()

    final_output = f"""
# Teaching Pack: {topic}

## 1. Concept Explanation

{concept.final_output}

---

## 2. Quiz

{quiz.final_output}

---

## 3. Lab Demo

{demo.final_output}

---

Execution Mode: Sequential  
Approximate Time Taken: {round(end_time - start_time, 2)} seconds
"""

    return final_output

# 7. Run Sequential Demo

Run the cell below and observe the time taken.


In [ ]:
sequential_output = await prepare_session_sequential("Parallel Multi-Agent Execution")

print(sequential_output)

## What did we just learn?

Sequential execution is simple.

But the Agents wait for each other even when their tasks are independent.

This can increase total response time.


# 8. Parallel Execution with `asyncio.gather`

Now we will run all three Agents at the same time.

Python provides a useful function:

```python
asyncio.gather(...)
```

It starts multiple async tasks together and waits until all are complete.

---

## Mental Model

```text
Start Concept Agent ─┐
Start Quiz Agent ────┼── Wait for all outputs
Start Demo Agent ────┘
```

Then combine the results.


In [ ]:
async def prepare_session_parallel(topic: str):
    """
    Runs specialist Agents in parallel using asyncio.gather.
    """

    start_time = time.time()

    concept_task = Runner.run(
        concept_agent,
        f"Explain {topic} in 5 simple bullet points."
    )

    quiz_task = Runner.run(
        quiz_agent,
        f"Create 3 MCQs with answers on {topic}."
    )

    demo_task = Runner.run(
        lab_demo_agent,
        f"Suggest one short notebook demo for {topic}."
    )

    concept, quiz, demo = await asyncio.gather(
        concept_task,
        quiz_task,
        demo_task
    )

    end_time = time.time()

    final_output = f"""
# Teaching Pack: {topic}

## 1. Concept Explanation

{concept.final_output}

---

## 2. Quiz

{quiz.final_output}

---

## 3. Lab Demo

{demo.final_output}

---

Execution Mode: Parallel  
Approximate Time Taken: {round(end_time - start_time, 2)} seconds
"""

    return final_output

# 9. Run Parallel Demo

Run the cell below and compare the approximate time with the sequential version.


In [ ]:
parallel_output = await prepare_session_parallel("Parallel Multi-Agent Execution")

print(parallel_output)

## What did we just learn?

Parallel execution can reduce total time when tasks are independent.

However, parallel execution should be used carefully.

It is useful when:

- Tasks are independent.
- Outputs do not depend on each other.
- Faster response is needed.
- API rate limits and cost are manageable.


# 10. Compare Sequential vs Parallel

Use this simple comparison.

| Factor | Sequential | Parallel |
|---|---|---|
| Execution style | One after another | At the same time |
| Best for | Dependent tasks | Independent tasks |
| Speed | Slower | Faster |
| Simplicity | Very simple | Slightly more advanced |
| Risk | Lower | Need error handling |
| Example | Draft → Review → Finalize | Research + Quiz + Demo |

---

## Important Teaching Point

Parallel execution is not always better.

It is better only when tasks are independent.


# 11. Add a Fourth Agent

Now let us extend the pattern.

We will add a **Case Study Agent**.

This Agent creates a short DSU-style classroom case study.


In [ ]:
case_study_agent = Agent(
    name="Case Study Agent",
    instructions="""
    You create short, realistic classroom case studies for engineering faculty.
    Use examples from colleges, classrooms, students, faculty, or enterprise systems.
    Keep the case study short and discussion-friendly.
    """,
)

# 12. Parallel Execution with Four Agents

Now the teaching pack will include:

1. Concept explanation
2. Quiz
3. Lab demo
4. Case study


In [ ]:
async def prepare_complete_session_parallel(topic: str):
    """
    Runs four specialist Agents in parallel.
    """

    start_time = time.time()

    concept_task = Runner.run(
        concept_agent,
        f"Explain {topic} in simple terms."
    )

    quiz_task = Runner.run(
        quiz_agent,
        f"Create 3 MCQs with answers on {topic}."
    )

    demo_task = Runner.run(
        lab_demo_agent,
        f"Suggest one short notebook demo for {topic}."
    )

    case_task = Runner.run(
        case_study_agent,
        f"Create one short classroom case study on {topic}."
    )

    concept, quiz, demo, case = await asyncio.gather(
        concept_task,
        quiz_task,
        demo_task,
        case_task
    )

    end_time = time.time()

    return f"""
# Complete FDP Session Pack: {topic}

## 1. Concept Explanation

{concept.final_output}

---

## 2. Classroom Quiz

{quiz.final_output}

---

## 3. Lab Demo Idea

{demo.final_output}

---

## 4. Case Study

{case.final_output}

---

Execution Mode: Parallel with Four Agents  
Approximate Time Taken: {round(end_time - start_time, 2)} seconds
"""

# 13. Run the Four-Agent Parallel Demo

This is a realistic teaching-assistant pattern.


In [ ]:
complete_pack = await prepare_complete_session_parallel("Agentic AI Guardrails")

print(complete_pack)

# 14. Enterprise Connection

Parallel multi-agent execution is useful in many enterprise scenarios.

## Example: Customer Escalation Analysis

```text
Customer Complaint
   ├── Sentiment Agent
   ├── Ticket History Agent
   ├── Policy Agent
   └── Resolution Recommendation Agent
   ↓
Combined escalation summary
```

## Example: DevOps Incident Review

```text
Production Incident
   ├── Log Analysis Agent
   ├── Metrics Agent
   ├── Deployment Agent
   └── RCA Drafting Agent
   ↓
Incident investigation summary
```

## Example: Academic FDP Preparation

```text
FDP Topic
   ├── Concept Agent
   ├── Quiz Agent
   ├── Demo Agent
   └── Case Study Agent
   ↓
Complete teaching pack
```


# 15. Error Handling Concept

In real systems, one parallel task may fail.

Example:

- Concept Agent succeeds.
- Quiz Agent succeeds.
- Demo Agent fails.

Should the full system fail?

Not always.

Enterprise systems usually need graceful handling.

For beginners, remember:

> Parallel execution needs better error handling than sequential execution.


In [ ]:
async def safe_runner(agent, prompt):
    """
    Simple safe wrapper for Agent execution.

    This prevents one failed Agent from breaking the entire workflow.
    """

    try:
        result = await Runner.run(agent, prompt)
        return result.final_output
    except Exception as e:
        return f"Agent failed with error: {str(e)}"

# 16. Parallel Execution with Basic Safety Wrapper

This pattern is useful for demos and early prototypes.


In [ ]:
async def prepare_session_parallel_safe(topic: str):
    """
    Runs Agents in parallel with basic error handling.
    """

    concept_task = safe_runner(
        concept_agent,
        f"Explain {topic} in simple terms."
    )

    quiz_task = safe_runner(
        quiz_agent,
        f"Create 3 MCQs with answers on {topic}."
    )

    demo_task = safe_runner(
        lab_demo_agent,
        f"Suggest one short notebook demo for {topic}."
    )

    concept, quiz, demo = await asyncio.gather(
        concept_task,
        quiz_task,
        demo_task
    )

    return f"""
# Safe Parallel Teaching Pack: {topic}

## Concept

{concept}

---

## Quiz

{quiz}

---

## Demo

{demo}
"""

In [ ]:
safe_output = await prepare_session_parallel_safe("Function Tools")

print(safe_output)

## What did we just learn?

When using parallel execution, always think about:

- What happens if one Agent fails?
- Can the system return a partial result?
- Should the user be told what failed?
- Should failed tasks be retried?

This is important for enterprise-ready Agent design.


# 17. Classroom Exercise

Design a parallel multi-agent system for:

> Preparing a student placement training session

Create four specialist Agents:

1. Resume Review Agent
2. Interview Question Agent
3. Coding Practice Agent
4. HR Tips Agent

Then run all four in parallel and combine the output.


In [ ]:
# Exercise Starter Code

resume_agent = Agent(
    name="Resume Review Agent",
    instructions="""
    TODO: Give resume improvement tips for engineering students.
    """,
)

interview_agent = Agent(
    name="Interview Question Agent",
    instructions="""
    TODO: Create beginner-friendly interview questions.
    """,
)

coding_practice_agent = Agent(
    name="Coding Practice Agent",
    instructions="""
    TODO: Suggest simple coding practice exercises.
    """,
)

hr_tips_agent = Agent(
    name="HR Tips Agent",
    instructions="""
    TODO: Give HR interview preparation tips.
    """,
)

# TODO:
# Use asyncio.gather to run all four Agents in parallel.


# 18. Challenge Exercise

Modify `prepare_complete_session_parallel()` to return a structured dictionary instead of one formatted string.

Example:

```python
{
    "topic": "...",
    "concept": "...",
    "quiz": "...",
    "demo": "...",
    "case_study": "..."
}
```

This is closer to how enterprise applications pass Agent outputs to front-end screens, APIs, or reports.


# 19. Common Mistakes

Avoid these mistakes:

1. Running dependent tasks in parallel.
2. Forgetting to use `await asyncio.gather(...)`.
3. Not handling failures in one parallel branch.
4. Creating too many Agents for simple tasks.
5. Ignoring API cost and rate limits.
6. Not combining outputs clearly.
7. Assuming parallel execution is always better.


# 20. Key Takeaways

In this notebook, we learned:

- Parallel execution runs multiple independent Agent tasks at the same time.
- Sequential execution is simpler but can be slower.
- `asyncio.gather` helps run multiple Agent calls concurrently.
- Parallel execution is useful for teaching packs, reports, investigations, and research workflows.
- Error handling becomes more important when tasks run in parallel.
- Use parallel execution only when tasks are independent.

---

## Final Mental Model

```text
Sequential:
Task A → Task B → Task C → Final Output

Parallel:
Task A ─┐
Task B ─┼→ Combine → Final Output
Task C ─┘
```

Parallel execution makes Agentic AI systems faster and more scalable when used correctly.
